# 11 · Reconocedor de placas de EasyOCR exportado a ONNX (lectura en el dispositivo)

Se ejecuta en **Google Colab** (CPU basta). EasyOCR no corre en el navegador, pero su **reconocedor** (VGG + BiLSTM + CTC, unos 15 MB) se puede exportar a ONNX y ejecutar con
`onnxruntime-web`, igual que el detector de placas. El detector de texto de EasyOCR (CRAFT) **no hace falta**: ya hay un detector de placas.

**Qué mide:**
1. **Franja a leer.** El recorte de la placa incluye el rótulo «BOLIVIA» (arriba) y los caracteres (abajo). Se prueba pasar al reconocedor el recorte entero o solo la parte inferior.
2. **Paridad PyTorch ↔ ONNX:** diferencia máxima entre salidas y si el texto leído es el mismo.
3. **Exactitud** sobre los 10 recortes reales, con el formato de A4 (`elegir_placa`), comparada con el 6/10 del notebook 10 (EasyOCR completo).
4. **Tamaño y velocidad.** Y si la cuantización dinámica (int8) conserva el resultado.

Sube `ml.zip` y `recortes.zip` (los mismos del notebook 10). Descarga `reconocedor_placas.onnx` + su ficha (fuera de git, I6: pesos al almacén S3) y `metrics.json`.

In [ ]:
import os, zipfile, json
from google.colab import files

subidos = files.upload()     # ml.zip + recortes.zip
os.makedirs("/content/condohub-ia", exist_ok=True)
os.makedirs("/content/recortes", exist_ok=True)
for nombre in subidos:
    z = zipfile.ZipFile(nombre)
    z.extractall("/content/condohub-ia" if nombre.lower().startswith("ml") else "/content/recortes")
os.chdir("/content/condohub-ia")
verdad = json.load(open("/content/recortes/verdad.json", encoding="utf-8"))
print(len(verdad), "recortes")

In [ ]:
!pip install -q easyocr onnx onnxruntime
import numpy as np, torch, torch.nn as nn, easyocr
lector = easyocr.Reader(["en"], gpu=False, quantize=False)   # en CPU EasyOCR cuantiza por defecto y eso no se exporta a ONNX
conversor = lector.converter
caracteres = list(conversor.character)          # el 0 es el «blank» de CTC

# En CPU EasyOCR cuantiza el reconocedor (y eso no se exporta a ONNX): se construye el modelo en coma flotante con los pesos originales.
from easyocr.model.vgg_model import Model     # el de english_g2 es VGG (el de model.model es ResNet)
estado = torch.load(os.path.expanduser("~/.EasyOCR/model/english_g2.pth"), map_location="cpu")
estado = {(k[7:] if k.startswith("module.") else k): v for k, v in estado.items()}      # quita el prefijo de DataParallel
modelo = Model(input_channel=1, output_channel=256, hidden_size=256, num_class=len(caracteres))
modelo.load_state_dict(estado)
modelo = modelo.eval()
cuantizadas = [n for n, m in modelo.named_modules() if "quantized" in type(m).__module__]
print("clases:", len(caracteres), "| capas cuantizadas:", len(cuantizadas), "| parámetros:", round(sum(p.numel() for p in modelo.parameters()) / 1e6, 1), "M")

## 1 · Exportar a ONNX

`AdaptiveAvgPool2d((None, 1))` no se exporta con ancho dinámico; equivale a promediar el último eje, así que se sustituye por eso (se comprueba la paridad después).

In [ ]:
class MediaUltimoEje(nn.Module):
    def forward(self, x):
        return x.mean(dim=3, keepdim=True)

if hasattr(modelo, "FeatureExtraction") and hasattr(modelo, "AdaptiveAvgPool"):
    modelo.AdaptiveAvgPool = MediaUltimoEje()

class Envoltura(nn.Module):
    def __init__(self, m):
        super().__init__(); self.m = m
    def forward(self, x):                           # x: [B, 1, 64, W] en -1..1  ->  probabilidades [B, T, clases]
        return torch.softmax(self.m(x, None), dim=2)

envoltura = Envoltura(modelo).eval()
ejemplo = torch.randn(1, 1, 64, 200)
with torch.no_grad():
    probas = envoltura(ejemplo)
print("salida de ejemplo:", tuple(probas.shape))
torch.onnx.export(envoltura, ejemplo, "/content/reconocedor_placas.onnx", input_names=["imagen"], output_names=["probabilidades"],
                  dynamic_axes={"imagen": {0: "lote", 3: "ancho"}, "probabilidades": {0: "lote", 1: "pasos"}}, opset_version=14, dynamo=False)
print("tamaño:", round(os.path.getsize("/content/reconocedor_placas.onnx") / 1e6, 1), "MB")

## 2 · Leer la placa: preproceso, decodificación CTC con lista de caracteres y franja inferior

In [ ]:
import onnxruntime as ort, time
from PIL import Image
from ml.src import placas

PERMITIDOS = set("0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ")
permitido = np.array([c in PERMITIDOS for c in caracteres])
sesion = ort.InferenceSession("/content/reconocedor_placas.onnx", providers=["CPUExecutionProvider"])

def preparar(ruta, desde):
    im = Image.open(ruta).convert("L")
    im = im.crop((0, int(desde * im.height), im.width, im.height))
    ancho = max(8, int(np.ceil(64 * im.width / im.height)))
    a = np.asarray(im.resize((ancho, 64), Image.BICUBIC), dtype=np.float32) / 255.0
    return ((a - 0.5) / 0.5)[None, None]                       # [1, 1, 64, W]

def decodificar(prob):
    """CTC voraz solo con los caracteres permitidos; confianza como la de EasyOCR (producto de máximos ** (2/√T))."""
    p = prob[0].copy()
    p[:, ~permitido & (np.arange(len(caracteres)) != 0)] = 0
    p = p / p.sum(axis=1, keepdims=True)
    idx, mx = p.argmax(axis=1), p.max(axis=1)
    texto, previo = "", 0
    for i in idx:
        if i != 0 and i != previo:
            texto += caracteres[i]
        previo = i
    return texto, float(np.prod(mx) ** (2.0 / np.sqrt(len(mx))))

def leer_onnx(ruta, desde, sesion=sesion):
    return decodificar(sesion.run(None, {"imagen": preparar(ruta, desde)})[0])

def leer_torch(ruta, desde):
    with torch.no_grad():
        return decodificar(envoltura(torch.from_numpy(preparar(ruta, desde))).numpy())

reales = [v["real"] for v in verdad]
rutas = [f"/content/recortes/{v['archivo']}" for v in verdad]
print("ejemplo:", leer_onnx(rutas[0], 0.4))

## 3 · ¿Qué franja de la placa conviene leer? Y paridad PyTorch ↔ ONNX

In [ ]:
resultados = {}
for desde in (0.0, 0.3, 0.4, 0.5):
    leidas_onnx = [leer_onnx(r, desde) for r in rutas]
    leidas_torch = [leer_torch(r, desde) for r in rutas]
    aceptadas = [placas.elegir_placa([(t, c)])[0] or "" for t, c in leidas_onnx]
    m = placas.metricas_lectura(reales, aceptadas)
    misma = sum(a[0] == b[0] for a, b in zip(leidas_onnx, leidas_torch))
    diff = max(abs(a[1] - b[1]) for a, b in zip(leidas_onnx, leidas_torch))
    resultados[f"desde_{desde}"] = {**m, "texto_igual_torch_onnx": f"{misma}/{len(rutas)}", "dif_max_confianza": diff,
                                    "rechazadas_por_formato": sum(a == "" for a in aceptadas)}
    print(f"desde {desde:.1f} de la altura: exactas {m['placa_completa'] * len(reales):.0f}/{len(reales)} | por carácter {m['precision_por_caracter']:.3f} | "
          f"rechazadas {resultados[f'desde_{desde}']['rechazadas_por_formato']} | texto igual PyTorch/ONNX {misma}/{len(rutas)} | dif. de confianza {diff:.4f}")
mejor = max(resultados, key=lambda k: (resultados[k]["placa_completa"], resultados[k]["precision_por_caracter"]))
print("\nmejor franja:", mejor)

## 3b · Votación entre franjas (la lectura que usará la PWA)

Probar todas las ventanas de 7 caracteres con corrección letra/dígito acepta «placas» desfasadas un carácter (falsas). Se lee desde varias alturas y una placa solo se acepta si aparece en
al menos 3 franjas (`ml/src/lectura_placas.py`). Se cuenta cuántas placas se aceptan **equivocadas**: es lo que importa para A4.

In [ ]:
from ml.src import lectura_placas

def evaluar_votos(minimo):
    textos = [(lectura_placas.elegir_por_votos([leer_onnx(r, f) for f in lectura_placas.FRANJAS], minimo)[0] or "") for r in rutas]
    exactas = sum(t == r for t, r in zip(textos, reales))
    equivocadas = sum(bool(t) and t != r for t, r in zip(textos, reales))
    print(f"votación, {minimo} franjas de {len(lectura_placas.FRANJAS)}: exactas {exactas}/{len(reales)} | equivocadas aceptadas {equivocadas} | rechazadas {sum(not t for t in textos)}")
    return {"exactas": exactas, "equivocadas_aceptadas": equivocadas, "rechazadas": sum(not t for t in textos)}

votacion = {f"minimo_{m}": evaluar_votos(m) for m in (2, 3)}

## 4 · Velocidad y cuantización dinámica int8

In [ ]:
desde = float(mejor.split("_")[1])
entrada = preparar(rutas[0], desde)
def medir(s, n=30):
    s.run(None, {"imagen": entrada})
    t = time.perf_counter()
    for _ in range(n):
        s.run(None, {"imagen": entrada})
    return (time.perf_counter() - t) / n * 1000

informe = {"fp32": {"MB": round(os.path.getsize("/content/reconocedor_placas.onnx") / 1e6, 2), "ms_cpu_colab": round(medir(sesion), 1)}}
try:
    from onnxruntime.quantization import quantize_dynamic, QuantType
    quantize_dynamic("/content/reconocedor_placas.onnx", "/content/reconocedor_placas_int8.onnx", weight_type=QuantType.QInt8)
    s8 = ort.InferenceSession("/content/reconocedor_placas_int8.onnx", providers=["CPUExecutionProvider"])
    leidas8 = [placas.elegir_placa([leer_onnx(r, desde, s8)])[0] or "" for r in rutas]
    m8 = placas.metricas_lectura(reales, leidas8)
    informe["int8_dinamico"] = {"MB": round(os.path.getsize("/content/reconocedor_placas_int8.onnx") / 1e6, 2), "ms_cpu_colab": round(medir(s8), 1),
                                "placa_completa": m8["placa_completa"], "precision_por_caracter": m8["precision_por_caracter"]}
except Exception as e:                                          # la cuantización dinámica puede no soportar alguna operación
    informe["int8_dinamico"] = {"error": f"{type(e).__name__}: {e}"[:300]}
print(json.dumps(informe, indent=2))

## 5 · Guardar y descargar

Ficha del modelo (caracteres, preproceso, franja) para que la PWA lo use igual; solo métricas en el repo, los pesos van al almacén S3 (I6).

In [ ]:
from google.colab import files
from ml.src.paths import dir_experimento
import hashlib

destino = dir_experimento("reconocedor_placas_onnx")
ficha = {"modelo": "easyocr-reconocedor-placas", "version": "1.0.0", "formato": "onnx-fp32", "archivo": "reconocedor_placas.onnx",
         "sha256": hashlib.sha256(open("/content/reconocedor_placas.onnx", "rb").read()).hexdigest(),
         "entrada": {"nombre": "imagen", "forma": ["lote", 1, 64, "ancho"],
                     "preproceso": f"gris, recortar desde el {int(desde * 100)} % de la altura hasta abajo, alto 64 con proporción (bicúbica), (x/255 - 0,5)/0,5"},
         "salida": {"nombre": "probabilidades", "forma": ["lote", "pasos", len(caracteres)], "significado": "softmax por paso; clase 0 = blank de CTC"},
         "caracteres": caracteres, "permitidos": "".join(sorted(PERMITIDOS)),
         "decodificacion": "CTC voraz restringido a los caracteres permitidos; confianza = producto de máximos ** (2/√pasos)"}
(destino / "ficha_reconocedor.json").write_text(json.dumps(ficha, indent=2, ensure_ascii=False), encoding="utf-8")
(destino / "metrics.json").write_text(json.dumps({"franjas": resultados, "mejor_franja": mejor, "votacion": votacion, "rendimiento": informe,
                                                   "comparacion": {"easyocr_completo_con_formato_A4": "6/10 (notebook 10)"}}, indent=2, ensure_ascii=False), encoding="utf-8")
for ruta in ("/content/reconocedor_placas.onnx", str(destino / "ficha_reconocedor.json"), str(destino / "metrics.json")):
    files.download(ruta)